# Chapter 4 – Numerical Modelling & Dynamical Assessment

**Project:** Floating Submerged Tunnel – Helsinki to Tallinn  
**Objective:** Determine the dynamic response of the SFT to wave, current, and seismic loading using frequency-domain and simplified time-domain methods.

---

## 4.1 Structural Model

The tunnel is modelled as a continuous beam on elastic tether supports.  A 2-D cross-section model is used for wave-induced motion analysis.

| Parameter | Symbol | Value | Unit |
|-----------|--------|-------|------|
| Outer diameter | D | TBD | m |
| Added mass coefficient | C_a | 1.0 | – |
| Drag coefficient | C_D | 1.0 | – |
| Tether stiffness (vertical) | k_v | TBD | kN/m |
| Tether stiffness (horizontal) | k_h | TBD | kN/m |
| Structure mass per unit length | m | TBD | kg/m |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal, linalg

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

# Physical constants
RHO_W = 1025.0
G = 9.81

## 4.2 Cross-Section Properties

In [ ]:
# -----------------------------------------------------------------------
# Tunnel cross-section (update with Chapter 2/3 values)
# -----------------------------------------------------------------------
D  = 14.0       # outer diameter [m]
A  = np.pi * D**2 / 4  # outer area [m²]
m  = 120e3      # structural mass per unit length [kg/m] (placeholder)
C_a = 1.0       # added mass coefficient
m_a = RHO_W * C_a * A  # added mass per unit length [kg/m]
m_total = m + m_a

# Tether stiffness (per unit length of tunnel)
L_tether = 50.0   # tether length [m]
L_span   = 30.0   # tether spacing [m]
E_steel  = 210e9  # Young's modulus [Pa]
d_tether = 0.150  # tether diameter [m]
A_tether = np.pi * d_tether**2 / 4

k_tether = E_steel * A_tether / L_tether  # [N/m] per tether
k_per_m  = k_tether / L_span              # [N/m²] per unit tunnel length

print(f"Added mass per unit length:  {m_a/1e3:.1f} t/m")
print(f"Total mass per unit length:  {m_total/1e3:.1f} t/m")
print(f"Tether stiffness per tether: {k_tether/1e6:.1f} MN/m")
print(f"Tether stiffness per m:      {k_per_m/1e3:.1f} kN/m/m")

## 4.3 Natural Frequencies (Single DOF – Heave)


In [ ]:
# -----------------------------------------------------------------------
# SDOF heave model (per unit length)
# Restoring: hydrostatic (negligible for deeply submerged) + tether stiffness
# -----------------------------------------------------------------------
k_heave = k_per_m   # [N/m per m of tunnel]
omega_n = np.sqrt(k_heave / m_total)  # natural circular frequency [rad/s]
f_n = omega_n / (2 * np.pi)           # natural frequency [Hz]
T_n = 1 / f_n                          # natural period [s]

# Critical damping ratio (structural + hydrodynamic radiation)
zeta = 0.05  # 5% assumed (placeholder – verify with model)

print(f"Natural circular frequency ω_n = {omega_n:.3f} rad/s")
print(f"Natural frequency f_n          = {f_n:.4f} Hz")
print(f"Natural period T_n             = {T_n:.2f} s")
print(f"Damping ratio ζ                = {zeta:.2f}")

## 4.4 Transfer Function (RAO – Response Amplitude Operator)


In [ ]:
# -----------------------------------------------------------------------
# SDOF frequency response (heave RAO)
# H(ω) = F(ω) / (k - mω² + i·cω)
# -----------------------------------------------------------------------
omega = np.linspace(0.01, 4.0, 500)   # [rad/s]
c_damp = 2 * zeta * np.sqrt(k_heave * m_total)  # [N·s/m per m]

# Wave force amplitude per unit length (Froude-Krylov + diffraction, simplified)
z_tunnel = -20.0   # depth [m]
F_wave_amp = RHO_W * G * A * np.exp(omega**2 / G * abs(z_tunnel))  # [N/m per m wave amplitude]

# Complex transfer function
H_complex = F_wave_amp / (k_heave - m_total * omega**2 + 1j * c_damp * omega)
RAO = np.abs(H_complex)  # [m/m]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(2 * np.pi / omega, RAO, color='steelblue', linewidth=1.5)
ax.axvline(T_n, color='tomato', linestyle='--', label=f'$T_n$ = {T_n:.1f} s')
ax.set_xlabel('Wave Period [s]')
ax.set_ylabel('Heave RAO [m/m wave amplitude]')
ax.set_title('Heave Response Amplitude Operator (SDOF)')
ax.set_xlim(1, 60)
ax.legend()
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## 4.5 Wave Spectrum & Response Spectrum


In [ ]:
# -----------------------------------------------------------------------
# JONSWAP spectrum (Gulf of Finland parameters)
# TODO: update Hs and Tp from Chapter 1 EVA
# -----------------------------------------------------------------------
def jonswap(omega, Hs, Tp, gamma=3.3):
    """JONSWAP wave energy spectrum S(ω) [m²·s/rad]."""
    omega_p = 2 * np.pi / Tp
    sigma = np.where(omega <= omega_p, 0.07, 0.09)
    alpha = 5 / 16 * Hs**2 * omega_p**4 / G**2 * (1 - 0.287 * np.log(gamma))
    r = np.exp(-0.5 * ((omega - omega_p) / (sigma * omega_p))**2)
    S_PM = alpha * G**2 / omega**5 * np.exp(-5 / 4 * (omega_p / omega)**4)
    return S_PM * gamma**r

Hs_100 = 5.0    # TODO: replace with EVA 100-yr value
Tp_100 = 9.0    # TODO: replace with EVA 100-yr value

omega_range = np.linspace(0.05, 4.0, 1000)
S_wave  = jonswap(omega_range, Hs_100, Tp_100)

# Interpolate RAO onto same omega grid
RAO_interp = np.interp(omega_range, omega, RAO)

# Response spectrum
S_response = RAO_interp**2 * S_wave

# Spectral moments
m0 = np.trapz(S_response, omega_range)
m2 = np.trapz(omega_range**2 * S_response, omega_range)
sigma_z = np.sqrt(m0)   # RMS heave response [m]
T_z = 2 * np.pi * np.sqrt(m0 / m2)  # zero-upcrossing period [s]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
ax1.plot(2 * np.pi / omega_range, S_wave, color='steelblue')
ax1.set_ylabel('$S_{wave}$ [m²·s/rad]')
ax1.set_title(f'JONSWAP Spectrum ($H_s$={Hs_100} m, $T_p$={Tp_100} s)')
ax1.grid(True, linestyle='--', alpha=0.4)

ax2.plot(2 * np.pi / omega_range, S_response, color='tomato')
ax2.set_ylabel('$S_{heave}$ [m²·s/rad]')
ax2.set_xlabel('Period [s]')
ax2.set_title(f'Heave Response Spectrum  (σ = {sigma_z:.3f} m,  $T_z$ = {T_z:.2f} s)')
ax2.grid(True, linestyle='--', alpha=0.4)

for ax in (ax1, ax2):
    ax.set_xlim(1, 60)

plt.tight_layout()
plt.show()

print(f"RMS heave response:       σ = {sigma_z:.4f} m")
print(f"Most probable extreme (3h storm):  {3.72 * sigma_z:.3f} m")

## 4.6 Summary & Recommendations

| Output | Value |
|--------|-------|
| Natural period T_n | *(fill in)* s |
| Damping ratio ζ | *(fill in)* |
| RMS heave (100-yr storm) | *(fill in)* m |
| MPM extreme heave (3 h) | *(fill in)* m |
| Max tether tension | *(fill in)* kN |

---
*Proceed to Chapter 5 – Reliability & Probabilistic Assessment*